# d1-Omni-2B — text, image, and audio decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/d1-Omni-2B-inference-decisions.ipynb)

Run Liquid AI's [`LiquidAI/d1-Omni-2B`](https://huggingface.co/LiquidAI/d1-Omni-2B) locally with `transformers`.
This is the ~2.17B decision head on [`LiquidAI/LFM3-Omni-2B`](https://huggingface.co/LiquidAI/LFM3-Omni-2B):
the same `system_one` API as d1-2B, plus **audio** clips (`{"type": "audio", ...}`).

It **replaces the older experimental d1-omni-600M** for new work. This notebook does **not** change
[`d1-omni-600M-inference-decisions.ipynb`](d1-omni-600M-inference-decisions.ipynb). Hub id spelling is **`d1-Omni-2B`** (capital O).

It is **not a chat model** and writes no transcript or speech. Without audio, the card says to prefer [`LiquidAI/d1-2B`](https://huggingface.co/LiquidAI/d1-2B).

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-Omni-2B/blob/main/LICENSE) (`license: other`, `lfm1.0`).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. Card default is bfloat16; this notebook uses **float16** (~4.3 GB) + **SDPA**. Audio is 16 kHz mono. Keep clips to a few seconds.

**Links:** [model card](https://huggingface.co/LiquidAI/d1-Omni-2B) · [LFM3 blog](https://www.liquid.ai/blog/lfm3) · [announcement](https://x.com/liquidai/status/2115128329003303107) · [collection](https://huggingface.co/collections/LiquidAI/lfm3) · [d1-2B notebook](d1-2B-inference-decisions.ipynb)


In [ ]:
%pip install -q -U "transformers>=5.19"
%pip install -q pillow torchvision soundfile

# Card pin: transformers>=5.19, torch, torchvision, pillow, trust_remote_code.
# soundfile reads the voice-note WAV. Do not reinstall torch.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.


## 1. Check the T4 and pick a dtype


In [ ]:
import os
import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.19"), (
    f"transformers {transformers.__version__} is loaded; this notebook needs 5.19+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# The cards default to bfloat16 on GPU. A T4 has no native BF16, so use float16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.float16
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")
if major >= 8:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


## 2. Load d1-Omni-2B


In [ ]:
from transformers import AutoModel

MODEL_ID = "LiquidAI/d1-Omni-2B"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, dtype=DTYPE).to(DEVICE)
model.eval()

print("parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print("dtype:", getattr(model, "dtype", DTYPE))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
)


`media` may list audio (`{"type": "audio", "path": ...}` or a waveform + `sampling_rate`), pictures, and video frames. A spoken question can be `{"type": "noul", "instructions": {"type": "audio", "path": "question.wav"}}`.


In [ ]:
def show(response):
    answers = response["answers"]
    for name, a in answers.items():
        kind = a.get("type")
        if kind == "noul":
            print(f"{name:>14}: P(yes) = {a['noul']:.3f}")
        elif kind == "choice" and "choices" in a:
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a.get("probabilities", {}).items())
            print(f"{name:>14}: {a['choices']}   [{probs}]")
        elif kind == "choice":
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a.get("probabilities", {}).items())
            print(f"{name:>14}: {a.get('choice')} ({a.get('confidence', 0):.2f})   [{probs}]")
        elif kind == "score":
            level = int(round(a["score"]))
            legend = a.get("legend", {})
            label = legend.get(str(level), legend.get(level, ""))
            print(f"{name:>14}: {a['score']:.2f} -> {label!r}")
        elif kind == "text":
            print(f"{name:>14}: {a.get('text')!r}  found={a.get('found')}")
        elif kind == "number":
            print(f"{name:>14}: {a.get('number')}  found={a.get('found')}")
        elif kind == "date":
            print(f"{name:>14}: {a.get('date')}  found={a.get('found')}")
        elif kind == "box":
            box = {k: round(v, 2) for k, v in (a.get("box") or {}).items()} if a.get("box") else None
            print(f"{name:>14}: {box}  found={a.get('found')}")
        elif kind == "boxes":
            print(f"{name:>14}: count={a.get('count')} boxes={a.get('boxes')}")
        else:
            print(f"{name:>14}: {a}")
    print(f"{'usage':>14}: {response.get('usage')}")


## 3. Demo 1 — text triage (same email as d1-2B / d1-3B)


In [ ]:
questions = {
    "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "shipping": "Delivery and parcels",
            "technical": "App or site faults",
        },
    },
    "topics": {
        "type": "choice",
        "multiple": True,
        "instructions": "Which topics does the message raise?",
        "criteria": {
            "billing": "Payments and charges",
            "delivery": "Shipping and parcels",
            "login": "Signing in to the account",
        },
    },
}
show(model.system_one(
    "I was charged twice and my parcel never arrived. Please refund one charge.",
    questions,
))


## 4. Demo 2 — image boxes (COCO cats)

Same photo as the d1-2B / d1-3B notebooks. Pass it through `media=`.


In [ ]:
from transformers.image_utils import load_image

image = load_image("http://images.cocodataset.org/val2017/000000039769.jpg")
display(image.resize((320, 240)))
show(model.system_one(
    None,
    {
        "cats": {
            "type": "number",
            "instructions": "How many cats are there?",
            "range": [0, 10],
            "step": 1,
        },
        "remote": {"type": "box", "instructions": "Where is the remote control on the left?"},
        "every_cat": {"type": "boxes", "instructions": "Every cat.", "max": 4},
    },
    media=[image],
))


## 5. Demo 3 — audio: beeps + a voice-note route

First, the card's synthetic 880 Hz beeps (no speech). Then a short English voice note synthesized with `espeak-ng` (same idea as the d1-omni-600M notebook: no recorded-voice licence). Audio is 16 kHz mono.


In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

rate = 16000
clip = np.random.default_rng(0).normal(0, 0.003, 4 * rate).astype(np.float32)
for start in (0.5, 1.5, 2.5):
    n = 3200
    clip[int(start * rate) : int(start * rate) + n] += (
        0.3 * np.sin(2 * np.pi * 880 * np.arange(n) / rate)
    )
print("=== card beep clip ===")
display(Audio(clip, rate=rate))
show(model.system_one(
    None,
    {
        "beeps": {
            "type": "number",
            "instructions": "How many beeps are there?",
            "range": [0, 10],
            "step": 1,
        },
        "speech": {"type": "noul", "instructions": "Does someone speak in the recording?"},
    },
    media=[{"type": "audio", "audio": clip, "sampling_rate": rate}],
))

spoken = (
    "Hey, can you cancel my flight to Chicago tomorrow and book a later one? "
    "The weather looks really bad."
)
wav_path = Path(tempfile.gettempdir()) / "d1-omni-2b-voice-note.wav"
exe = shutil.which("espeak-ng") or shutil.which("espeak")
if exe is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
    exe = shutil.which("espeak-ng") or shutil.which("espeak")
subprocess.run([exe, "-s", "145", "-v", "en-us", "-w", str(wav_path), spoken], check=True)
audio, sr = sf.read(wav_path, dtype="float32")
if audio.ndim > 1:
    audio = audio.mean(axis=1)
if sr != rate:
    n_out = max(1, int(round(len(audio) * rate / sr)))
    t_in = np.linspace(0.0, 1.0, num=len(audio), endpoint=False)
    t_out = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
    audio = np.interp(t_out, t_in, audio.astype(np.float64)).astype(np.float32)
print()
print("=== voice note ===")
print(spoken)
display(Audio(audio, rate=rate))
show(model.system_one(
    "Voice note from a user talking to an assistant.",
    {
        "topic": {
            "type": "choice",
            "instructions": "What is the speaker talking about?",
            "criteria": {
                "travel": "Travel, flights, or transport",
                "billing": "Charges, invoices, or refunds",
                "food": "Food, groceries, or meals",
                "weather": "Only the weather, with no travel request",
            },
        },
        "intent": {
            "type": "choice",
            "instructions": "What does the speaker want the assistant to do?",
            "criteria": {
                "rebook": "Change or replace an existing booking",
                "cancel_only": "Cancel and stop, with no replacement",
                "status": "Check a reservation or a delay",
                "other": "Something else",
            },
        },
        "refund": {"type": "noul", "instructions": "Is the speaker asking for a refund?"},
    },
    media=[{"type": "audio", "audio": audio, "sampling_rate": rate}],
))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
)


## Caveats

- **Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-Omni-2B/blob/main/LICENSE) (`license: other`).
- **Hub id:** `LiquidAI/d1-Omni-2B` (capital O). The 600M sibling is `LiquidAI/d1-omni-600M` and stays in its own notebook.
- **T4 caps:** float16, short 16 kHz clips. Long spoken passages lose accuracy (card). Prefer d1-2B when there is no audio.
- **HF token:** public weights. Cancel Colab `HF_TOKEN` prompts.
- **Not executed here:** no GPU in the agent VM. Full T4 `system_one` was not run.
- **Do not merge from this agent.**
